# Lecture 3 — From Theory to Practice: Live Demo of Foundation Models with Hugging Face

**Presented by Dr. Fitsum Assamnew Andargie**  
**Duration:** 90 minutes  
**Environment:** Google Colab + Hugging Face

> **Central question:** What actually happens between writing an input and receiving an intelligent-looking output from a foundation model?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fassamnew/Foundation-Models/blob/main/Lecture_3_Foundation_Models_Hugging_Face_Colab.ipynb)

**Repository:** [github.com/fassamnew/Foundation-Models](https://github.com/fassamnew/Foundation-Models)

## Learning outcomes

By the end of this lab, participants should be able to:

1. distinguish **architecture, learned parameters, data, and hardware**;
2. load and use pretrained models from the **Hugging Face Hub**;
3. trace **text → tokens → representations → attention → prediction**;
4. distinguish **pretraining, inference, and fine-tuning**;
5. complete a short **gradient-based fine-tuning** experiment;
6. run a compact **generative language model**;
7. demonstrate **multimodal zero-shot classification** with CLIP; and
8. discuss **African-language coverage, compute access, licensing, bias, local data, and sovereignty**.

## 90-minute route

| Time | Activity |
|---|---|
| 0–7 min | Setup + hardware |
| 7–15 min | Pretrained model inference |
| 15–25 min | Self-supervised pretraining intuition |
| 25–38 min | Tokens, hidden states, attention |
| 38–55 min | Micro fine-tuning |
| 55–70 min | Generative model |
| 70–80 min | Multimodal CLIP |
| 80–87 min | African-language context |
| 87–90 min | Responsible deployment + wrap-up |

### Teaching pattern

For each demo use: **PREDICT → RUN → INSPECT → MODIFY → EXPLAIN**.

# 0. Setup

## Enable a GPU in Colab

A GPU makes model loading, fine-tuning, and generation much faster. Do this **before** running the install and setup cells below.

### Step 1 — Open the runtime settings

1. In the Colab menu bar at the top, click **Runtime**.
2. In the dropdown, click **Change runtime type**.

![Step 1: Runtime → Change runtime type](https://raw.githubusercontent.com/fassamnew/Foundation-Models/main/images/colab-gpu-step1-runtime-menu.png)

### Step 2 — Select a GPU and save

1. In the **Change runtime type** dialog, keep **Runtime type** as **Python 3**.
2. Open the **Hardware accelerator** dropdown and choose a GPU option (often **T4 GPU**; any available GPU is fine).
3. Click **Save**.

Colab may reconnect the session after you save. That is expected.

![Step 2: Hardware accelerator → GPU, then Save](https://raw.githubusercontent.com/fassamnew/Foundation-Models/main/images/colab-gpu-step2-change-runtime.png)

### Step 3 — Confirm the GPU is active

Check either of these:

- Top-right resource indicator: you should see **GPU** (not only RAM / Disk).
- Or open **Runtime → View resources** and confirm a GPU is listed.

Then run the setup cell below. You want output like:

```text
CUDA available: True
GPU: Tesla T4   # name may differ
```

If you see `CUDA available: False`, repeat Steps 1–2 and re-run the setup cell.

![Step 3: Confirm GPU is connected](https://raw.githubusercontent.com/fassamnew/Foundation-Models/main/images/colab-gpu-step3-verify-gpu.png)

### If no GPU is available

Free Colab sometimes has no GPU capacity. You can still follow the notebook on CPU; the micro-training section already uses smaller settings when a GPU is missing. Training and generation will be slower.

## Install libraries

The notebook uses current Hugging Face interfaces. Before the workshop, run this notebook once in a fresh Colab runtime and record the package versions that worked. You can then pin those versions for maximum reproducibility.

In [ ]:
%pip -q install -U transformers datasets accelerate huggingface_hub scikit-learn matplotlib pillow

In [ ]:
import gc, platform, random, time
import numpy as np
import torch, transformers, datasets
from transformers import set_seed

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('Datasets:', datasets.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")
else:
    print('CPU runtime detected; smaller training settings will be used.')

## Where is the intelligence?

At this point we have code and hardware, but no model has been loaded.

**AI system = model architecture + learned parameters + runtime hardware + input data**

This distinction connects the demo to the hardware section of the main lecture. A Transformer architecture is not a trained foundation model, and a trained model still requires software and hardware to execute.

**Ask the room:** Which components change during fine-tuning?

In [ ]:
def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def gpu_memory():
    if not torch.cuda.is_available():
        print('GPU memory: not available')
        return
    print(f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB | Reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB")

gpu_memory()

# 1. USE — A pretrained model through `pipeline`

A Hugging Face `pipeline` provides a high-level inference interface. We begin with DistilBERT already fine-tuned on SST-2 sentiment classification.

**Predict before running:** Which examples should be positive, negative, or difficult to classify?

In [ ]:
from transformers import pipeline

SENTIMENT_MODEL = 'distilbert/distilbert-base-uncased-finetuned-sst-2-english'
classifier = pipeline('text-classification', model=SENTIMENT_MODEL,
                      device=0 if torch.cuda.is_available() else -1)

examples = [
    'This workshop is extremely useful.',
    'The system failed and the experience was frustrating.',
    'The model worked, although the result was not as useful as I expected.'
]
for text in examples:
    print(text)
    print(classifier(text))
    print('-'*70)

## What happened?

The pipeline performed:

**text → tokenization → tensors → Transformer → classification head → label**

We did **not** train the model. We downloaded learned parameters and performed **inference**.

### Try it yourself

Change an input to a sentence from agriculture, health, education, engineering, or your own research domain. Compare a clear statement with an ambiguous one.

**Discussion:** A high model score is not the same as certainty or truth. What validation would be required before deployment?

# 2. PRETRAINING — Self-supervised learning intuition

BERT-family models use **masked language modeling** as a core pretraining idea: hide a token and predict plausible replacements from context. This demonstrates how learning signals can be created from raw text without manually labeling every example.

In [ ]:
BASE_MODEL = 'distilbert/distilbert-base-uncased'
fill_mask = pipeline('fill-mask', model=BASE_MODEL,
                     device=0 if torch.cuda.is_available() else -1)

for text in [
    'Artificial intelligence can help farmers [MASK] crop diseases.',
    'A foundation model can be adapted to many [MASK].'
]:
    print('\nINPUT:', text)
    for item in fill_mask(text, top_k=5):
        print(f"{item['token_str']!r:15s} score={item['score']:.4f}")

## Why this matters

```text
Large unlabeled corpus
        ↓
Self-supervised pretraining
        ↓
Reusable pretrained model
        ↓
 ┌────────────┬─────────────┬──────────────┐
 ↓            ↓             ↓
Sentiment   QA / NER     Other tasks
```

That reusability is central to the **foundation-model** idea.

### Try it yourself

Replace a word in your own sentence with `[MASK]`. Ask whether the outputs are syntactically plausible, semantically plausible, and whether any assumptions or biases are visible.

# 3. OPEN — Tokens and language representation

Transformers do not directly receive sentences. They receive **token IDs**. Tokenization is therefore part of the model system and can influence both efficiency and language coverage.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
sentence = 'Foundation models learn reusable representations.'
enc = tokenizer(sentence, return_tensors='pt')

print('Text:', sentence)
print('Tokens:', tokenizer.tokenize(sentence))
print('Input IDs:', enc['input_ids'])
print('Attention mask:', enc['attention_mask'])
print('IDs as tokens:', tokenizer.convert_ids_to_tokens(enc['input_ids'][0]))

## Quick local-language check

The model above is an **English uncased DistilBERT** model. Compare English with Amharic. This is not a complete multilingual benchmark; it is a compact demonstration that tokenizer vocabulary is itself part of representation capacity.

In [ ]:
texts = {
    'English': 'Artificial intelligence can support agriculture.',
    'Amharic': 'ሰው ሰራሽ አስተዋይነት ግብርናን መደገፍ ይችላል።'
}
for language, text in texts.items():
    pieces = tokenizer.tokenize(text)
    print(f'\n{language}')
    print('Text:', text)
    print('Tokens:', pieces)
    print('Number of token IDs:', len(tokenizer(text)['input_ids']))

**Discuss:** What happens downstream when a language is poorly represented in the tokenizer and pretraining corpus? Consider inefficient tokenization, weak representations, language mixing, missing cultural concepts, and unequal performance.

# 4. OPEN THE TRANSFORMER — Hidden states and attention

We now bypass the pipeline and call the Transformer directly. The model produces a contextual representation for each token, and each attention layer contains multiple attention heads.

In [ ]:
from transformers import AutoModel

encoder = AutoModel.from_pretrained(BASE_MODEL)
encoder.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder.to(device)

text = 'Foundation models learn reusable representations.'
inputs = tokenizer(text, return_tensors='pt')
inputs = {k:v.to(device) for k,v in inputs.items()}

with torch.no_grad():
    outputs = encoder(**inputs, output_hidden_states=True,
                      output_attentions=True, return_dict=True)

print('Last hidden-state shape:', tuple(outputs.last_hidden_state.shape))
print('Interpretation: batch × tokens × hidden dimension')
print('Transformer layers:', len(outputs.hidden_states)-1)
print('Attention tensors:', len(outputs.attentions))
print('One attention tensor:', tuple(outputs.attentions[-1].shape))
print('Interpretation: batch × heads × query tokens × key tokens')

## Visualize attention

For readability, average the final layer across attention heads. Treat this as an **attention-weight visualization**, not as a complete explanation of the model's reasoning or a causal feature-importance method.

In [ ]:
import matplotlib.pyplot as plt

token_labels = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
att = outputs.attentions[-1][0].mean(dim=0).detach().cpu().numpy()

plt.figure(figsize=(9,7))
plt.imshow(att, aspect='auto')
plt.xticks(range(len(token_labels)), token_labels, rotation=75)
plt.yticks(range(len(token_labels)), token_labels)
plt.xlabel('Key token')
plt.ylabel('Query token')
plt.title('Final-layer self-attention averaged across heads')
plt.colorbar(label='Attention weight')
plt.tight_layout()
plt.show()

### Try it yourself

Compare `The bank approved the loan.` with `We sat on the bank of the river.`

**Question:** Would you expect the contextual representation of `bank` to be identical in both sentences? Why not?

# 5. ADAPT — Micro fine-tuning in a few minutes

This is the workshop's actual training exercise.

We are **not training a foundation model from scratch**. We start from pretrained DistilBERT and adapt it to labeled sentiment data.

To keep the exercise time-bounded, we use:

- a small SST-2 subset;
- maximum sequence length of 96 tokens;
- one epoch; and
- **partial fine-tuning**: embeddings and the first four DistilBERT layers are frozen; the last two Transformer layers and classification head remain trainable.

This is genuine gradient-based training, but it uses substantially less compute than full fine-tuning.

> **Teaching question:** Why can a pretrained model adapt from hundreds of labeled examples when training from scratch would require vastly more data?

In [ ]:
from datasets import load_dataset

TRAIN_N, VAL_N = (600, 200) if torch.cuda.is_available() else (160, 80)
raw_train = load_dataset('nyu-mll/glue', 'sst2', split='train')
raw_val = load_dataset('nyu-mll/glue', 'sst2', split='validation')
train_ds = raw_train.shuffle(seed=SEED).select(range(TRAIN_N))
val_ds = raw_val.shuffle(seed=SEED).select(range(VAL_N))

print('Training examples:', len(train_ds))
print('Validation examples:', len(val_ds))
print('Example:', train_ds[0])
print('Labels: 0=negative, 1=positive')

In [ ]:
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding

MAX_LENGTH = 96

def tokenize_batch(batch):
    return tokenizer(batch['sentence'], truncation=True, max_length=MAX_LENGTH)

tokenized_train = train_ds.map(tokenize_batch, batched=True)
tokenized_val = val_ds.map(tokenize_batch, batched=True)

train_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2,
    id2label={0:'NEGATIVE',1:'POSITIVE'},
    label2id={'NEGATIVE':0,'POSITIVE':1}
)

# Fast workshop mode: update only the top of the encoder + classification head.
train_model.distilbert.embeddings.requires_grad_(False)
for layer in train_model.distilbert.transformer.layer[:4]:
    layer.requires_grad_(False)

total = sum(p.numel() for p in train_model.parameters())
trainable = sum(p.numel() for p in train_model.parameters() if p.requires_grad)
print(f'Total parameters: {total:,}')
print(f'Trainable parameters: {trainable:,}')
print(f'Trainable fraction: {100*trainable/total:.2f}%')

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None
)

## What is being trained?

```text
Pretrained DistilBERT encoder
        ↓
Task-specific classification head
        ↓
NEGATIVE / POSITIVE
```

The task-specific classification head starts newly initialized. Most of the linguistic knowledge comes from pretraining. In this fast configuration, only the upper part of the encoder is allowed to adapt.

In [ ]:
from transformers import Trainer, TrainingArguments

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': float((preds == labels).mean())}

args = TrainingArguments(
    output_dir='./workshop_distilbert_sst2',
    learning_rate=5e-5,
    per_device_train_batch_size=16 if torch.cuda.is_available() else 8,
    per_device_eval_batch_size=32 if torch.cuda.is_available() else 16,
    num_train_epochs=1,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=10,
    report_to='none',
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

trainer = Trainer(
    model=train_model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
print('Trainer ready.')

## Baseline before training

The classification head has just been initialized. Before running, ask what random guessing would achieve on a balanced two-class problem.

In [ ]:
before = trainer.evaluate()
print(f"Before training accuracy: {before['eval_accuracy']:.3f}")
print(f"Before training loss: {before['eval_loss']:.3f}")

## Train and measure the wall-clock time

The exact time and accuracy will vary with the assigned Colab hardware and package versions. The goal is to experience **adaptation**, not to maximize benchmark performance.

In [ ]:
start = time.perf_counter()
train_result = trainer.train()
elapsed = time.perf_counter() - start
print(f'Training time: {elapsed:.1f} seconds ({elapsed/60:.2f} minutes)')

In [ ]:
after = trainer.evaluate()
print('Before training accuracy:', round(before['eval_accuracy'],3))
print('After training accuracy: ', round(after['eval_accuracy'],3))
print('After training loss:     ', round(after['eval_loss'],3))

## Interpret the result

```text
PRETRAINING                     FINE-TUNING
Broad text data                 Small labeled task data
      ↓                               ↓
General representations  →  Task-adapted representations
```

We did **not** create the language knowledge in this session. We reused it.

If time permits, increase the dataset, unfreeze more layers, or add another epoch and compare **accuracy, training time, and compute cost**.

# 6. GENERATE — A compact instruction-tuned language model

A causal language model predicts the next token repeatedly, enabling open-ended generation. We use **SmolLM2-360M-Instruct** because it is small enough for a workshop runtime while still demonstrating modern chat-style generation.

In [ ]:
del trainer, train_model, encoder
cleanup(); gpu_memory()

In [ ]:
GEN_MODEL = 'HuggingFaceTB/SmolLM2-360M-Instruct'
generator = pipeline('text-generation', model=GEN_MODEL,
                     device_map='auto', dtype='auto')

messages = [{
    'role':'user',
    'content':'Give three practical ways artificial intelligence could support a smallholder farmer. Keep the answer concise.'
}]

out = generator(messages, max_new_tokens=120, do_sample=False)
print(out[0]['generated_text'][-1]['content'])

## Generation settings are not training

At inference time the model repeatedly produces a probability distribution over possible next tokens. `do_sample=False` is more deterministic; sampling allows alternative continuations. Changing decoding settings does **not** alter the model weights.

In [ ]:
out2 = generator(messages, max_new_tokens=120,
                 do_sample=True, temperature=0.9, top_p=0.9)
print(out2[0]['generated_text'][-1]['content'])

## Connect parameters to hardware

The next calculation estimates storage for **weights only**. Actual inference and especially training require additional memory for activations, caches, optimizer states, temporary tensors, and framework overhead.

In [ ]:
params = sum(p.numel() for p in generator.model.parameters())
print(f'Parameters: {params/1e6:.1f} million')
for bits in [32,16,8,4]:
    print(f'{bits:>2}-bit weights ≈ {params*bits/8/1e9:.3f} GB')

This connects directly to mixed precision, quantization, memory-efficient attention, distributed training, and accelerator design. Modern foundation models are **systems-engineering achievements**, not architecture alone.

# 7. MULTIMODAL — CLIP zero-shot image classification

CLIP learns related representations for **images and text**. We can therefore provide candidate text descriptions at inference time without training a new fixed classifier.

> **Timing note:** this requires another model download. If workshop bandwidth is weak, make this instructor-led or assign it as a post-session extension.

In [ ]:
del generator
cleanup(); gpu_memory()

In [ ]:
from PIL import Image
import requests
from io import BytesIO
from IPython.display import display

url = 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/coco_sample.png'
r = requests.get(url, timeout=30); r.raise_for_status()
image = Image.open(BytesIO(r.content)).convert('RGB')
display(image)

In [ ]:
CLIP_MODEL = 'openai/clip-vit-base-patch32'
clip_classifier = pipeline('zero-shot-image-classification', model=CLIP_MODEL,
                           device=0 if torch.cuda.is_available() else -1)
labels = [
    'a photograph of animals',
    'a photograph of people',
    'a photograph of a city',
    'a photograph of agricultural land',
    'a photograph taken indoors'
]
results = clip_classifier(image, candidate_labels=labels)
for item in results:
    print(f"{item['label']:<40s} {item['score']:.4f}")

## What is the key idea?

We did not train a five-class image classifier. The model compares learned image and text representations. This illustrates the move from **task-specific representations** to **reusable multimodal representations**.

### Try your own image

```python
from google.colab import files
uploaded = files.upload()
path = next(iter(uploaded))
my_image = Image.open(path).convert('RGB')
display(my_image)
clip_classifier(my_image, candidate_labels=['...', '...'])
```

Use images you have permission to use.

# 8. AFRICAN CONTEXT — Inspect before you deploy

A globally popular model is not automatically the most appropriate model for every language or country. We will inspect **InkubaLM-0.4B** metadata without first downloading the full model.

This is also good operational practice: **read the model card before executing unfamiliar model code**.

In [ ]:
from huggingface_hub import model_info

AFRICAN_MODEL = 'lelapa/InkubaLM-0.4B'
info = model_info(AFRICAN_MODEL)
card = info.card_data.to_dict() if info.card_data is not None else {}

print('Model:', info.id)
print('License:', card.get('license'))
print('Languages:', card.get('language'))
print('Pipeline tag:', info.pipeline_tag)
print('Tags:', info.tags[:15] if info.tags else None)

## Questions for an Ethiopian deployment

1. Is **Amharic** represented? What about Afaan Oromo, Tigrinya, Somali, Sidama, Afar, or other required languages?
2. What pretraining data shaped the model?
3. What license applies?
4. What evidence exists for the **exact intended task**?
5. What local evaluation is still necessary?
6. Can the model run economically on available infrastructure?
7. Who controls the data, model hosting, updates, and logs?

> **Open model ≠ locally appropriate model.**

### Optional instructor extension

InkubaLM's repository currently uses remote custom model code. Only use `trust_remote_code=True` after reviewing and trusting that repository.

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
model_id = 'lelapa/InkubaLM-0.4B'
inkuba_tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
inkuba_model = AutoModelForCausalLM.from_pretrained(
    model_id, trust_remote_code=True, device_map='auto', dtype='auto'
)
```

This is a useful place to discuss model openness, software supply-chain trust, licensing, data transparency, and deployment suitability.

# 9. RESPONSIBLE DEPLOYMENT CHALLENGE

Imagine an institution wants an AI assistant that gives crop-disease guidance to farmers in multiple Ethiopian languages. A promising model exists on Hugging Face.

**Can we deploy it?** Review the model against the following dimensions.

| Dimension | Question |
|---|---|
| Task fit | Was it evaluated for this task? |
| Language | Does it represent the intended languages and varieties? |
| Data | What relevant local data is absent? |
| Reliability | How will hallucinations and uncertainty be evaluated? |
| Human oversight | Which outputs require expert review? |
| Safety | What happens when advice is wrong? |
| Bias | Which populations or regions may be poorly represented? |
| Privacy | What user or institutional data enters the system? |
| Sovereignty | Where are data and inference processed? |
| Compute | Can it run reliably and affordably? |
| License | Is the intended use permitted? |
| Monitoring | How will failures be detected after deployment? |

## Final systems view

```text
                 FOUNDATION MODEL
                       │
       ┌───────────────┼────────────────┐
       │               │                │
     DATA          ARCHITECTURE       COMPUTE
       └───────────────┼────────────────┘
                       ↓
                  PRETRAINING
                       ↓
             Reusable representations
                       ↓
       domain adaptation + local data
                       +
                   evaluation
                       +
                human oversight
                       +
              responsible governance
                       ↓
                DEPLOYED AI SYSTEM
```

> **Main takeaway:** A foundation model is not the finished AI system. It is a computational foundation from which a system can be built.

# 10. What we did

### USE
`pretrained model → input → output`

### OPEN
`text → tokenizer → token IDs → representations → attention → Transformer → prediction`

### ADAPT
`pretrained model + small labeled dataset → gradient-based fine-tuning → downstream model`

We then connected these ideas to **generation, multimodality, African-language representation, compute, and responsible deployment**.

# 11. Post-workshop challenges

1. **Compare tokenizers:** test English, Amharic, Afaan Oromo, or another language across English-only and multilingual models.
2. **Full vs partial fine-tuning:** unfreeze all DistilBERT layers and compare time, memory, and validation accuracy.
3. **Parameter-efficient fine-tuning:** explore LoRA/PEFT and compare trainable parameter counts.
4. **Domain evaluation:** build a small, carefully reviewed evaluation set from your domain without exposing confidential data.
5. **Model-card audit:** assess intended use, languages, data, license, risks, compute, and local validation requirements for a Hugging Face model.

# 12. References and resources

## Selected academic references

- Rumelhart, Hinton & Williams (1986) — Backpropagation.
- Krizhevsky, Sutskever & Hinton (2012) — AlexNet / ImageNet.
- Mikolov et al. (2013) — word2vec.
- Bahdanau, Cho & Bengio (2014) — Attention for NMT.
- Jouppi et al. (2017) — TPU.
- Micikevicius et al. (2017) — Mixed precision training.
- Vaswani et al. (2017) — *Attention Is All You Need*.
- Devlin et al. (2018) — BERT.
- Radford et al. (2018) — GPT.
- Shoeybi et al. (2019) — Megatron-LM.
- Brown et al. (2020) — GPT-3.
- Dosovitskiy et al. (2020) — Vision Transformer.
- Kaplan et al. (2020) — Scaling laws.
- Rajbhandari et al. (2020) — ZeRO.
- Wolf et al. (2020) — Transformers library.
- Bommasani et al. (2021) — Foundation models.
- Jumper et al. (2021) — AlphaFold2.
- Radford et al. (2021) — CLIP.
- Avsec et al. (2021) — Enformer.
- Hoffmann et al. (2022) — Chinchilla.
- Dao et al. (2022) — FlashAttention.
- Lam et al. (2023) — GraphCast.
- Merchant et al. (2023) — GNoME.
- Singhal et al. (2023) — Med-PaLM.
- Jakubik et al. (2023) — Prithvi geospatial FM.
- Lin et al. (2023) — ESM-2.
- Tonja et al. (2024) — InkubaLM.

## Hugging Face resources used

- https://huggingface.co/docs/transformers/
- https://huggingface.co/docs/datasets/
- https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english
- https://huggingface.co/distilbert/distilbert-base-uncased
- https://huggingface.co/datasets/nyu-mll/glue
- https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct
- https://huggingface.co/openai/clip-vit-base-patch32
- https://huggingface.co/lelapa/InkubaLM-0.4B

## GitHub hosting

This notebook is hosted at:

- **Repo:** https://github.com/fassamnew/Foundation-Models
- **Open in Colab:** https://colab.research.google.com/github/fassamnew/Foundation-Models/blob/main/Lecture_3_Foundation_Models_Hugging_Face_Colab.ipynb

Repository structure:

```text
Foundation-Models/
├── images/
│   ├── colab-gpu-step1-runtime-menu.png
│   ├── colab-gpu-step2-change-runtime.png
│   └── colab-gpu-step3-verify-gpu.png
└── Lecture_3_Foundation_Models_Hugging_Face_Colab.ipynb
```

Before delivery:

1. run the notebook from a fresh Colab GPU runtime;
2. note the package versions printed at startup;
3. optionally pin those exact versions in the installation cell; and
4. retain an already-executed instructor copy as a fallback for poor connectivity.